# Verificação de Novos Arquivos do Novo Caged

Este notebook verifica, no FTP público do PDET/MTPS (`ftp://ftp.mtps.gov.br/pdet/microdados/NOVO%20CAGED/`), se foi publicado um novo arquivo `CAGEDMOVAAAAMM.7z` no **ano vigente** que ainda não foi ingerido na camada Bronze.

Fluxo:

1. Lista os arquivos `CAGEDMOV*.7z` do ano vigente no FTP
2. Compara com as competências já ingeridas na Bronze e com os arquivos já presentes no volume
3. Baixa os novos arquivos para o volume `raw_novo_caged`, conferindo o tamanho
4. Registra o resultado em um arquivo de controle no volume (`_controle/novas_competencias.txt`): a lista das novas competências, ou o marcador `NENHUMA` quando não há novidades — na próxima tarefa do job, o notebook `01_landing_to_bronze` ingere somente as competências listadas ou encerra o job sem ingerir

In [0]:
import ftplib
import os
import re
from datetime import datetime

In [0]:
# Servidor FTP público do Ministério do Trabalho (PDET)
host_ftp = "ftp.mtps.gov.br"
caminho_ftp = "/pdet/microdados/NOVO CAGED"

# Ano vigente (a verificação é feita apenas no ano corrente)
ano_vigente = str(datetime.now().year)

# Volume com os arquivos brutos já baixados (.7z)
path = "/Volumes/workspace/bronze/raw_novo_caged"

# Tabela Bronze com as competências já ingeridas
tabela_bronze = "workspace.bronze.novo_caged_movimentacoes"

# A próxima tarefa do job (ingestão) é o notebook 01_landing_to_bronze,
# que lê o arquivo de controle gravado na última célula

In [0]:
# Conecta ao FTP (login anônimo) e lista os arquivos CAGEDMOV*.7z publicados
# no ano vigente. Cada competência fica em uma subpasta:
# .../NOVO CAGED/2026/202601/CAGEDMOV202601.7z
ftp = ftplib.FTP()
ftp.connect(host_ftp, 21, timeout=30)
ftp.login()
ftp.encoding = "latin-1"  # a listagem do servidor usa latin-1

try:
    ftp.cwd(f"{caminho_ftp}/{ano_vigente}")
except ftplib.error_perm:
    print(f"O diretório do ano {ano_vigente} ainda não existe no FTP.")
    arquivos_ftp = []
else:
    arquivos_ftp = []
    for pasta in sorted(ftp.nlst()):
        # Considera apenas as subpastas de competência (AAAAMM)
        if not re.fullmatch(r"\d{6}", pasta):
            continue
        for nome in ftp.nlst(pasta):
            resultado = re.match(r"CAGEDMOV(\d{6})\.7z$", nome, re.IGNORECASE)
            if resultado:
                arquivos_ftp.append({
                    "competencia": resultado.group(1),
                    "nome_arquivo": nome,
                    "caminho_remoto": f"{caminho_ftp}/{ano_vigente}/{pasta}/{nome}",
                    "tamanho_bytes": ftp.size(f"{pasta}/{nome}"),
                })
    arquivos_ftp.sort(key=lambda arq: arq["competencia"])

assert len(arquivos_ftp) > 0, (
    f"Nenhum arquivo CAGEDMOV encontrado no FTP para o ano {ano_vigente}."
)

print(f"Arquivos no FTP para {ano_vigente}: {len(arquivos_ftp)}")
for arq in arquivos_ftp:
    print(f"  {arq['nome_arquivo']}  {arq['tamanho_bytes'] / 1024 / 1024:.1f} MB")

In [0]:
# Competências já ingeridas na Bronze (se a tabela existir)
tabela_existe = (
    spark.sql("SHOW TABLES IN workspace.bronze LIKE 'novo_caged_movimentacoes'")
    .count() > 0
)

if tabela_existe:
    competencias_ingeridas = {
        linha["_competencia_referencia"]
        for linha in spark.table(tabela_bronze)
        .select("_competencia_referencia")
        .distinct()
        .collect()
    }
else:
    competencias_ingeridas = set()

# Arquivos CAGEDMOV*.7z já presentes no volume local
arquivos_locais = set()
for raiz, _, nomes in os.walk(path):
    if raiz.startswith(f"{path}/_extracao"):
        continue
    for nome in nomes:
        resultado = re.match(r"CAGEDMOV(\d{6})\.7z$", nome, re.IGNORECASE)
        if resultado:
            arquivos_locais.add(resultado.group(1))

# Novas competências: publicadas no FTP do ano vigente e ainda não ingeridas
novos_arquivos = [
    arq for arq in arquivos_ftp
    if arq["competencia"] not in competencias_ingeridas
]

# Das novas, quais precisam ser baixadas (ainda não estão no volume)
arquivos_para_baixar = [
    arq for arq in novos_arquivos
    if arq["competencia"] not in arquivos_locais
]

print(f"Competências já ingeridas na Bronze: {len(competencias_ingeridas)}")
print(f"Novas competências a ingerir: {len(novos_arquivos)}")
print(f"Arquivos a baixar do FTP: {len(arquivos_para_baixar)}")
for arq in novos_arquivos:
    situacao = (
        "já presente no volume (somente ingerir)"
        if arq["competencia"] in arquivos_locais
        else "baixar do FTP"
    )
    print(f"  {arq['nome_arquivo']}: {situacao}")

In [0]:
# Baixa os novos arquivos .7z do FTP para o volume raw,
# mantendo a estrutura por ano (ex.: .../2026/CAGEDMOV202609.7z)
dir_ano_local = f"{path}/{ano_vigente}"
os.makedirs(dir_ano_local, exist_ok=True)

for arq in arquivos_para_baixar:
    destino_local = f"{dir_ano_local}/{arq['nome_arquivo']}"
    print(f"Baixando {arq['nome_arquivo']} ({arq['tamanho_bytes'] / 1024 / 1024:.1f} MB)...")

    with open(destino_local, "wb") as arquivo_local:
        ftp.retrbinary(f"RETR {arq['caminho_remoto']}", arquivo_local.write)

    # Confere o tamanho do arquivo baixado contra o servidor
    tamanho_local = os.path.getsize(destino_local)
    assert tamanho_local == arq["tamanho_bytes"], (
        f"{arq['nome_arquivo']}: tamanho baixado ({tamanho_local:,} bytes) "
        f"difere do servidor ({arq['tamanho_bytes']:,} bytes)."
    )
    print(f"  Baixa concluída e tamanho conferido: {tamanho_local:,} bytes")

In [0]:
# Encerra a conexão FTP
try:
    ftp.quit()
except Exception:
    pass

# Registra o resultado da verificação em um arquivo de controle, lido pela
# próxima tarefa do job (01_landing_to_bronze) para decidir o que ingerir:
#   - lista de competências (ex.: "202609") -> o 01 ingere somente elas
#   - marcador NENHUMA -> nada novo; o 01 encerra o job sem ingerir
arquivo_controle = f"{path}/_controle/novas_competencias.txt"
os.makedirs(os.path.dirname(arquivo_controle), exist_ok=True)

# Atenção: nunca gravar um arquivo vazio — em volumes serverless a
# sobrescrita com conteúdo vazio não substitui o arquivo anterior
resultado_verificacao = (
    ",".join(arq["competencia"] for arq in novos_arquivos)
    if len(novos_arquivos) > 0
    else "NENHUMA"
)

with open(arquivo_controle, "w", encoding="utf-8") as arquivo:
    arquivo.write(resultado_verificacao)

# Confirma na hora o que ficou gravado no volume
tamanho_gravado = os.path.getsize(arquivo_controle)
print(f"Arquivo de controle atualizado: {arquivo_controle}")
print(
    f"Conteúdo registrado: {resultado_verificacao!r} "
    f"({tamanho_gravado} bytes no disco)"
)

if len(novos_arquivos) > 0:
    print(
        "Novas competências registradas: "
        + ", ".join(arq["competencia"] for arq in novos_arquivos)
    )
    print("A próxima tarefa do job (01_landing_to_bronze) fará a ingestão.")
else:
    print("Nenhum arquivo novo encontrado no FTP.")
    print("A próxima tarefa do job (01_landing_to_bronze) encerrará sem ingerir.")